In [ ]:
## login wandb
import wandb
wandb.login()
## set up project name
import os
os.environ["WANDB_PROJECT"] = "chess-llm" 
os.environ["UNSLOTH_VLLM_STANDBY"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

In [ ]:
import unsloth
import vllm
import torch
import trl

print(vllm.__version__)
print(unsloth.__version__)
print(torch.__version__)
print(trl.__version__)

## Model

In [ ]:
from unsloth import FastLanguageModel

max_seq_length = 1024 # Can increase for longer reasoning traces
lora_rank = 128 # Larger rank = smarter, but slower

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Norrawee/Qwen3-4B-Thinking-2507-exp05", 
    max_seq_length = max_seq_length,
    load_in_4bit = False, # False for LoRA 16bit
    fast_inference = False, # Enable vLLM fast inference
    max_lora_rank = lora_rank,
    gpu_memory_utilization = 0.9, # Reduce if out of memory
)

In [ ]:
from util import add_chess_tokens

add_chess_tokens(model, tokenizer)

In [ ]:
xs = tokenizer("<a1><White_Pawn><a2><White_Knight>")
print([tokenizer.decode(x) for x in xs["input_ids"]])

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = lora_rank*2, # *2 speeds up training
    use_gradient_checkpointing = "unsloth", # Reduces memory usage
    random_state = 3407,
)

## Data

In [ ]:
from datasets import load_dataset, Dataset
from tqdm import tqdm
import chess
import pandas as pd

In [ ]:
SYSTEM_PROMPT = """You are an expert chess player.
Task:  
- Given a chess position, select the single best move.  
- Analyze the position objectively and compare 1–3 candidate moves.  
- Consider tactics, strategy, king safety, material balance, and positional factors.  
- Conclude with the strongest move only.
- Return **only** the chosen move in UCI format enclosed by <uci_move> and </uci_move>.

Position: {board}
Moves: {legal_moves_uci_list}
Your turn: {side_to_move}"""

In [ ]:
from datasets import load_dataset  
  
dataset = load_dataset("Norrawee/sft-exp05")
df = dataset["train"].to_pandas()
df = df[:100]

In [ ]:
from util import encode_board_position

def format_prompt(row):
    board, moves = encode_board_position(row["FEN"])
    prompt = SYSTEM_PROMPT.format(
        side_to_move=row["side_to_move"],
        legal_moves_uci_list=moves,
        board=board,
    ) 
    response = f"<think>\n{row["explanation"]}\n</think>\n\n<uci_move>{row["target_move"]}</uci_move>"   
    return [  
        {"role": "user", "content": prompt},  
        {"role": "assistant", "content": response},  
    ]

In [ ]:
## preprocess
df["prompt"] = df.apply(format_prompt, axis=1)
df["text"] = tokenizer.apply_chat_template(df["prompt"].values.tolist(), tokenize=False)

In [ ]:
from sklearn.model_selection import train_test_split  
from datasets import Dataset  
  
# Unique boards  
unique_boards = df["FEN"].unique()  
  
# Split boards, NOT rows  
train_boards, test_boards = train_test_split(  
    unique_boards,  
    test_size=0.1 if len(df) < 500 else 50,  
    random_state=42,  
    shuffle=True,  
)  
  
# Filter rows  
train_df = df[df["FEN"].isin(train_boards)].reset_index(drop=True)  
test_df  = df[df["FEN"].isin(test_boards)].reset_index(drop=True)  
  
# Create HF datasets  
ds = {  
    "train": Dataset.from_pandas(train_df),  
    "test": Dataset.from_pandas(test_df),  
}


In [ ]:
text = ds["train"]["text"][0]

print(text)
print(len(tokenizer(text)["input_ids"]))

## SFT

In [ ]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset= ds["train"],
    eval_dataset= ds["test"],
    args = SFTConfig(
        dataset_text_field = "text",
        optim = "adamw_8bit",
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "wandb", # Use TrackIO/WandB etc
        
        # training params
        learning_rate=5e-5,
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        gradient_accumulation_steps=4,
        # num_train_epochs=5,
        fp16=False,
        bf16=True,
        weight_decay = 0.001,
        
        # logging
        # eval_strategy="epoch",
        # save_strategy="epoch",
        # logging_strategy="epoch",
        eval_strategy="steps",
        save_strategy="steps",
        logging_strategy="steps",
        logging_steps=10,
        save_steps=10,
        eval_steps=10,
        save_total_limit=1,
        max_steps=30,
    ),
)

In [ ]:
trainer.train()

## Test

In [ ]:
# import re
# UCI_PATTERN = re.compile(r"<uci_move>(.*?)</uci_move>")  
  
# def extract_uci(text):  
#     match = UCI_PATTERN.search(text)  
#     return match.group(1).strip() if match else None  

In [ ]:
# import re
# UCI_PATTERN = re.compile(r"<uci_move>(.*?)</uci_move>")  
# def extract_uci(text):  
#     match = UCI_PATTERN.search(text)  
#     return match.group(1).strip() if match else None  
 
# import chess  
# import chess.engine  
# # ---------------- CONFIG ----------------  
# ENGINE_PATH = "stockfish"   # change if needed  
# ENGINE_LIMIT = chess.engine.Limit(time=1.0, depth=16) 

  
# def evaluate(fen_board, uci_move, verbose=False):
#     try:
#         engine = chess.engine.SimpleEngine.popen_uci(ENGINE_PATH)
#         board = chess.Board(fen_board)  

#         ## before 
#         info_before = engine.analyse(board, ENGINE_LIMIT)  
#         eval_before = info_before["score"].relative.score(mate_score=1000) 

#         move = chess.Move.from_uci(uci_move) 
#         board.push(move)  

#         ## after
#         info_after = engine.analyse(board, ENGINE_LIMIT)  
#         eval_after = - info_after["score"].relative.score(mate_score=1000)
        
#         engine.quit()
        
#         if verbose:
#             print(eval_before, eval_after)
        
#         ## calculate reward
#         delta = eval_after - eval_before
#         if delta > 100:
#             return 2
#         elif delta > -100:
#             return 1
#         else:
#             return 0
#     except:
#         print("Error")
#         return -1
  


In [ ]:
# def generate_batch(model, prompts):  
#     texts = [  
#         tokenizer.apply_chat_template(p, tokenize=False, add_generation_prompt=True)  
#         for p in prompts  
#     ]  
  
#     model_inputs = tokenizer(  
#         texts,  
#         return_tensors="pt",  
#         padding=True,  
#     ).to(model.device)  
  
#     generated = model.generate(  
#         **model_inputs,  
#         max_new_tokens=200,
#         do_sample=True,
#         temperature=0.01,
#         top_p=0.8,
#         top_k=20,
#     )   
  
#     decoded = tokenizer.batch_decode(  
#         generated,  
#         skip_special_tokens=True  
#     )  
  
#     return [t.split("assistant")[-1].strip() for t in decoded]  

In [ ]:
# from tqdm import tqdm  
  
# BATCH_SIZE = 1 ## padding affects the outputs (i dont know how to fix).
  
# responses = []
# for batch_start in tqdm(range(0, len(ds["test"]), BATCH_SIZE)):  
#     batch = ds["test"][batch_start: batch_start + BATCH_SIZE]  
  
#     prompts = [ex[:1] for ex in batch["prompt"]]  
#     responses.extend(generate_batch(model, prompts))

In [ ]:
# outputs = []

# for i, (response, example) in enumerate(zip(responses, ds["test"])):
#     pred_move = extract_uci(response)  
#     legal_moves_uci_list = example["legal_moves_uci_list"]
#     fen = example["FEN"]

#     if pred_move not in legal_moves_uci_list:
#         legal = False
#         score = -1000
#     else:
#         legal = True
#         score = evaluate(fen, pred_move)
#     outputs.append({
#         "score": score,
#         "response": response,
#         "legal": legal
#     })
    
#     # if i < 10:
#     #     print(example["board_utf"])
#     #     print(response)
#     #     print(score)
#     #     print(f"Target move: {example["target_move"]}")
#     #     print("*"*60)


In [ ]:
# output_df = pd.DataFrame(outputs)
# output_df[["legal"]].value_counts()

In [ ]:
# output_df[["score"]].describe()

## Save

In [ ]:
# model.push_to_hub_merged(
#     "Norrawee/Qwen/Qwen2.5-7B-Instruct-sft-exp04", 
#     tokenizer,
#     save_method = "merged_16bit", 
# )

In [ ]:
# wandb.finish()